# Multimodal Lecture Summarization - Backend Production Pipeline & Quality Gate Testbed

Notebook này được tích hợp **chuẩn quy trình Backend hiện tại** của hệ thống, đồng thời bổ sung các bộ **Quality Gate & Cross-Verification** để giải quyết triệt để 3 vấn đề cốt lõi:
- **[VẤN ĐỀ 5] Audio ASR Quality Gate**: Kiểm tra chất lượng âm thanh, độ tin cậy segment, WER proxy, phát hiện silence và mismatch ngôn ngữ.
- **[VẤN ĐỀ 7] Visual Pipeline Quality Gate**: Phát hiện keyframe mờ nhòe (blur detection) và lọc trùng slide thông minh kết hợp OCR (tránh mất slide quan trọng có cùng layout/template).
- **[VẤN ĐỀ 8] Caption Hallucination Verification**: Kiểm tra đối chiếu Caption (VLM) với OCR Evidence (PaddleOCR) để chống suy diễn sai (hallucination) trước khi đưa vào LLM.

### Input mặc định
Chạy trên **TED Talk video** tại D:\datasets\TEDLIUM\videos\ (TED-LIUM companion videos, tiếng Anh). Đổi TED_VIDEO_NAME ở ô Input Setup nếu muốn talk khác.

### Qui trình Pipeline Backend mở rộng:
1. **Setup Environment & Worker Config**
2. **Stage 1: Audio Extraction & ASR** (`AudioTranscriber` - Faster-Whisper)
3. **Stage 1B [VẤN ĐỀ 5]: Audio ASR Quality Gate & Confidence Validator**
4. **Stage 2: Speaker Diarization** (`SpeakerDiarizer` - pyannote)
5. **Stage 3: Visual Scene Detection** (`SceneDetector` - PySceneDetect)
6. **Stage 4: Visual Semantic Analysis** (`SemanticAnalyzer` - CLIP + Florence-2 + PaddleOCR)
7. **Stage 4B [VẤN ĐỀ 7]: Visual Quality Gate & OCR-Aware Smart Slide Deduplication**
8. **Stage 4C [VẤN ĐỀ 8]: Caption Hallucination Verification & Evidence Grounding**
9. **Stage 5: Multimodal Timeline Alignment** (`TimelineBuilder`)
10. **Stage 6: Evidence-Grounded LLM Summarization** (`Summarizer`)
11. **Stage 7: Backend Result Alignment**
12. **Stage 8: Backend Database & Vector Ingestion Simulation**
13. **Stage 9: Integrated Quality Assurance & Image Retrieval Suite**

## 1. Setup Environment & Worker Configuration

In [ ]:
import os
import sys
import json
import time
import warnings
import uuid
from pathlib import Path

# Resolve repo root by markers. Prefer the notebook path (Cursor/VS Code)
# because Path("../..") + os.chdir breaks on re-run of this cell.
def _find_project_root() -> Path:
    starts: list[Path] = []
    nb = globals().get("__vsc_ipynb_file__")
    if nb:
        starts.append(Path(nb).resolve().parent)
    starts.append(Path.cwd())
    seen: set[Path] = set()
    for start in starts:
        for p in [start, *start.parents]:
            rp = p.resolve()
            if rp in seen:
                continue
            seen.add(rp)
            if (rp / "ai_workers").is_dir() and (rp / "experiments").is_dir():
                return rp
    raise FileNotFoundError(
        "Could not find project root. Expected a directory containing "
        "ai_workers/ and experiments/. Restart the kernel if cwd is wrong."
    )

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Ép dùng model local trong ./cache, tránh tải lại HuggingFace/Xet
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

# Stability: nếu máy thiếu/không load được một số cuDNN ops -> kernel có thể chết.
# Ép chạy CPU để có thể chạy so sánh backend vs thực nghiệm.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")

os.environ["HF_HOME"] = str(PROJECT_ROOT / "cache" / "huggingface")
os.environ["HUGGINGFACE_HUB_CACHE"] = str(PROJECT_ROOT / "cache" / "huggingface" / "hub")
os.environ["TORCH_HOME"] = str(PROJECT_ROOT / "cache" / "torch_hub")
os.chdir(PROJECT_ROOT)  # để worker_settings.CACHE_DIR="./cache" resolve đúng

warnings.filterwarnings("ignore")

# Nạp Worker Settings chính thức từ backend
from ai_workers.core.config import worker_settings
print(f"Project Root: {PROJECT_ROOT}")
print(f"CWD: {os.getcwd()}")
print(f"HF_HOME: {os.environ['HF_HOME']}")
print(f"Cache Directory: {worker_settings.CACHE_DIR}")
print(f"LLM Provider: {worker_settings.LLM_PROVIDER} ({worker_settings.LLM_MODEL})")

# Kiểm tra PyTorch & CUDA status
import torch
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
# Khi ép CUDA_VISIBLE_DEVICES="" có thể vẫn báo is_available nhưng device_count=0
if torch.cuda.is_available() and torch.cuda.device_count() > 0:
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 2. Input Setup & Output Directory (TED-LIUM Video + Mock Backend Job ID)

Mặc định lấy video TED từ `D:\datasets\TEDLIUM\videos\`. Đổi `TED_VIDEO_NAME` nếu muốn talk khác (ví dụ Barry Schwartz, Ken Robinson). Fallback về `demo_data/sample.mp4` nếu thư mục TED không có file.


In [ ]:
# Tạo Job ID giả lập giống Celery Task
JOB_ID = f"ted_job_{uuid.uuid4().hex[:8]}"

# ===== Pipeline mode switches =====
# True  -> chạy thực nghiệm mở rộng (QA gates 5/7/8 + cleanup trước timeline/LLM)
# False -> bám sát backend production hiện tại trong ai_workers/tasks.py
ENABLE_EXPERIMENTAL_GATES = False
ENABLE_UPSTREAM_UTTERANCE_CLEANUP = False
# Stage 9.4 (CLIP retrieval) is extra; disable during parity runs to avoid extra GPU memory.
ENABLE_RETRIEVAL_TESTS = False

# Academic tuning knobs for experimental runs
MIN_UTTERANCE_MERGE_DUR_SEC = 1.5
MIN_CHAPTER_DURATION_SEC = 45.0
MIN_KEYFRAME_KEEP_RATIO = 0.25

RUN_MODE = "experimental_enhanced" if (ENABLE_EXPERIMENTAL_GATES or ENABLE_UPSTREAM_UTTERANCE_CLEANUP) else "backend_parity"
# Override JOB_ID to include mode (so we can compare outputs easily)
JOB_ID = f"{RUN_MODE}_job_{uuid.uuid4().hex[:8]}"

# --- TED-LIUM companion videos (Drive D) ---
TED_VIDEO_DIR = Path(r"D:\datasets\TEDLIUM\videos")
# Short smoke-test default (~2.6 min). Alternatives:
#   "Barry_Schwartz.mp4"
#   "Do schools kill creativity? | Sir Ken Robinson | TED.mp4"
#   "The paradox of choice | Barry Schwartz | TED.mp4"
TED_VIDEO_NAME = "Blaise_Agueray_Arcas.mp4"
EXPECTED_LANG = "en"  # TED Talks are English

ted_candidate = TED_VIDEO_DIR / TED_VIDEO_NAME
demo_fallback = PROJECT_ROOT / "experiments" / "notebooks" / "demo_data" / "sample.mp4"

if ted_candidate.is_file():
    VIDEO_PATH = str(ted_candidate)
    INPUT_SOURCE = "TED-LIUM videos"
elif TED_VIDEO_DIR.is_dir():
    mp4s = sorted(TED_VIDEO_DIR.glob("*.mp4"), key=lambda p: p.stat().st_size)
    if mp4s:
        VIDEO_PATH = str(mp4s[0])
        INPUT_SOURCE = "TED-LIUM videos (auto shortest)"
        print(f"[WARN] '{TED_VIDEO_NAME}' not found; using shortest TED video: {mp4s[0].name}")
    else:
        VIDEO_PATH = str(demo_fallback)
        INPUT_SOURCE = "demo fallback"
        print("[WARN] No TED mp4 found; falling back to demo sample.mp4")
else:
    VIDEO_PATH = str(demo_fallback)
    INPUT_SOURCE = "demo fallback"
    print(f"[WARN] TED dir missing ({TED_VIDEO_DIR}); falling back to demo sample.mp4")

OUTPUT_DIR = str(PROJECT_ROOT / "outputs" / JOB_ID)
os.makedirs(OUTPUT_DIR, exist_ok=True)

available = sorted(p.name for p in TED_VIDEO_DIR.glob("*.mp4")) if TED_VIDEO_DIR.is_dir() else []
print(f"Job ID: {JOB_ID}")
print(f"Input Source: {INPUT_SOURCE}")
print(f"Expected Language: {EXPECTED_LANG}")
print(f"Video Input: {VIDEO_PATH}")
print(f"Exists: {os.path.exists(VIDEO_PATH)} | Size: {os.path.getsize(VIDEO_PATH)/1e6:.1f} MB" if os.path.exists(VIDEO_PATH) else "Exists: False")
print(f"Output Dir: {OUTPUT_DIR}")
print(f"Experimental Gates: {ENABLE_EXPERIMENTAL_GATES}")
print(f"Upstream Utterance Cleanup: {ENABLE_UPSTREAM_UTTERANCE_CLEANUP}")
if available:
    print(f"Available TED videos ({len(available)}):")
    for name in available:
        print(f"  - {name}")


## 3. Stage 1: Audio Extraction & Transcription (Faster-Whisper)

In [ ]:
from ai_workers.modules.audio_v2.transcriber import AudioTranscriber

print("=== STAGE 1: AUDIO TRANSCRIBER ===")
audio_transcriber = AudioTranscriber()
audio_result = audio_transcriber.process(VIDEO_PATH)

print(f"Detected Language: {audio_result.get('language')}")
print(f"Full Text Length: {len(audio_result.get('text', ''))} chars")
print(f"Total Segments: {len(audio_result.get('segments', []))}")
for seg in audio_result.get('segments', [])[:3]:
    print(f"  [{seg['start']:.2f}s -> {seg['end']:.2f}s]: {seg['text']}")

### 3.1 [VẤN ĐỀ 5] Audio ASR Quality Gate & Confidence Validator
*Mục đích*: Đánh giá độ tin cậy của transcript đầu vào trước khi truyền cho các stage downstream. Tránh lỗi dây chuyền do ASR hỏng, audio rỗng, hoặc phát hiện ngôn ngữ sai.

In [ ]:
def validate_audio_asr(audio_res: dict, expected_lang: str = "en", min_conf_thresh: float = 0.6) -> dict:
    """
    Quality Gate cho Audio ASR Stage (Giải quyết Vấn đề 5):
    - Pre-check duration & audio extraction
    - Confidence score check per segment
    - Language mismatch detection
    - Silence / No-speech ratio check
    - Generates warning/error flags & fallback recommendations
    """
    segments = audio_res.get("segments", [])
    full_text = audio_res.get("text", "").strip()
    detected_lang = audio_res.get("language", "unknown")

    report = {
        "total_segments": len(segments),
        "total_chars": len(full_text),
        "detected_language": detected_lang,
        "language_mismatch": False,
        "low_confidence_segments": 0,
        "empty_segments": 0,
        "no_speech_ratio": 0.0,
        "average_confidence": 1.0,
        "quality_status": "HIGH",
        "visual_dominant_fallback": False,
        "recommended_mode": "audio_visual_balanced",
        "asr_weight": 0.75,
        "warnings": []
    }

    # 1. Check empty audio / extraction failure
    if not segments or len(full_text) == 0:
        report["quality_status"] = "FAILED"
        report["visual_dominant_fallback"] = True
        report["recommended_mode"] = "visual_only_emergency"
        report["asr_weight"] = 0.10
        report["warnings"].append("CRITICAL: Audio transcript is empty or audio extraction failed!")
        return report

    # 2. Check language mismatch
    if expected_lang and detected_lang != expected_lang:
        report["language_mismatch"] = True
        report["warnings"].append(f"Language Mismatch Warning: Detected '{detected_lang}', expected '{expected_lang}'.")

    # 3. Analyze segment confidence & silence
    conf_scores = []
    for seg in segments:
        text = seg.get("text", "").strip()
        seg["low_confidence_flag"] = False

        if not text:
            report["empty_segments"] += 1
            continue

        # Calculate segment confidence proxy from word probabilities or avg_logprob if available
        words = seg.get("words", [])
        if words:
            seg_conf = sum(w.get("probability", 0.9) for w in words) / len(words)
        else:
            seg_conf = 0.85  # Default fallback score

        seg["asr_confidence"] = round(float(seg_conf), 3)
        conf_scores.append(seg_conf)

        if seg_conf < min_conf_thresh:
            report["low_confidence_segments"] += 1
            seg["low_confidence_flag"] = True

    if conf_scores:
        report["average_confidence"] = float(sum(conf_scores) / len(conf_scores))

    report["no_speech_ratio"] = report["empty_segments"] / max(1, len(segments))
    low_ratio = report["low_confidence_segments"] / max(1, len(segments))

    if low_ratio > 0.40 or report["average_confidence"] < 0.65:
        report["quality_status"] = "LOW_QUALITY_WARNING"
        report["visual_dominant_fallback"] = True
        report["recommended_mode"] = "visual_dominant"
        report["asr_weight"] = 0.35
        report["warnings"].append(f"High ratio of low-confidence ASR segments ({low_ratio*100:.1f}%). Triggering Visual-Dominant Fallback mode.")
    elif low_ratio > 0.15:
        report["quality_status"] = "MEDIUM"
        report["recommended_mode"] = "audio_visual_balanced"
        report["asr_weight"] = 0.55

    if report["no_speech_ratio"] > 0.25:
        report["warnings"].append(f"High no-speech ratio detected ({report['no_speech_ratio']*100:.1f}%).")

    return report


def build_asr_safe_segments(segments: list, report: dict, min_chars: int = 3) -> list:
    """Lọc các segment ASR quá rủi ro để tránh kéo sai cho downstream."""
    if not segments:
        return []

    safe_segments = []
    for seg in segments:
        text = seg.get("text", "").strip()
        if len(text) < min_chars:
            continue

        if report.get("visual_dominant_fallback") and seg.get("low_confidence_flag"):
            continue

        safe_segments.append(seg)

    # Nếu lọc quá mạnh làm rỗng dữ liệu thì fallback lại dữ liệu gốc để không gãy pipeline
    return safe_segments if safe_segments else segments


# Chạy Quality Gate cho Audio ASR (TED Talks = English)
if ENABLE_EXPERIMENTAL_GATES:
    audio_quality_report = validate_audio_asr(audio_result, expected_lang=EXPECTED_LANG)
    audio_result["safe_segments"] = build_asr_safe_segments(audio_result.get("segments", []), audio_quality_report)
else:
    # Parity mode: bám sát backend tasks.py (không lọc segment theo quality gate)
    audio_quality_report = {
        "total_segments": len(audio_result.get("segments", [])),
        "total_chars": len(audio_result.get("text", "").strip()),
        "detected_language": audio_result.get("language", "unknown"),
        "language_mismatch": False,
        "low_confidence_segments": 0,
        "empty_segments": 0,
        "no_speech_ratio": 0.0,
        "average_confidence": 1.0,
        "quality_status": "BYPASSED_BACKEND_PARITY",
        "visual_dominant_fallback": False,
        "recommended_mode": "backend_default",
        "asr_weight": 1.0,
        "warnings": ["Audio ASR quality gate is bypassed in backend parity mode."]
    }
    audio_result["safe_segments"] = audio_result.get("segments", [])

print("=== [VẤN ĐỀ 5] AUDIO ASR QUALITY GATE REPORT ===")
print(f"Quality Status: {audio_quality_report['quality_status']}")
print(f"Average Confidence: {audio_quality_report['average_confidence']:.2f}")
print(f"Low-Confidence Segments: {audio_quality_report['low_confidence_segments']}/{audio_quality_report['total_segments']}")
print(f"No-Speech Ratio: {audio_quality_report['no_speech_ratio']:.2f}")
print(f"Recommended Mode: {audio_quality_report['recommended_mode']} | ASR Weight: {audio_quality_report['asr_weight']}")
print(f"Safe Segments kept: {len(audio_result['safe_segments'])}/{len(audio_result.get('segments', []))}")
if audio_quality_report["warnings"]:
    for w in audio_quality_report["warnings"]:
        print(f"  [WARNING] {w}")

## 4. Stage 2: Speaker Diarization

In [ ]:
from ai_workers.modules.audio_v2.speaker import SpeakerDiarizer

print("=== STAGE 2: SPEAKER DIARIZATION ===")
speaker_diarizer = SpeakerDiarizer()
audio_wav_path = VIDEO_PATH.rsplit(".", 1)[0] + ".wav"

segments_for_diarization = audio_result.get("safe_segments", audio_result.get("segments", []))
utterances = speaker_diarizer.process(
    audio_wav_path,
    segments_for_diarization
)

print(f"ASR segments passed into diarization: {len(segments_for_diarization)}")
print(f"Total Utterances with Speaker Labels: {len(utterances)}")
for utt in utterances[:5]:
    print(f"  [{utt.get('speaker', 'SPEAKER')}] ({utt['start']:.2f}s -> {utt['end']:.2f}s): {utt['text']}")

## 5. Stage 3: Visual Scene Detection & Keyframe Extraction

In [ ]:
from ai_workers.modules.visual_v2.scene_detector import SceneDetector

print("=== STAGE 3: SCENE DETECTOR ===")
scene_detector = SceneDetector()
visual_result = scene_detector.process(VIDEO_PATH, OUTPUT_DIR)

scenes = visual_result.get("scenes", [])
print(f"Total Extracted Keyframes/Scenes: {len(scenes)}")
for sc in scenes[:3]:
    print(f"  Scene {sc.get('scene_index')}: {sc.get('start_timecode')} -> {sc.get('end_timecode')}, Path: {sc.get('keyframe_path')}")

## 6. Stage 4: Visual Semantic Analysis (CLIP Filtering + Florence-2 Captioning + PaddleOCR)

Thành phần `SemanticAnalyzer` hoạt động theo 3 bước:
1. **CLIP Filtering (`filter_scenes_clip`)**: Lọc bớt logo, chuyển cảnh graphic và các frame trùng lặp.
2. **Florence-2 Captioning (`caption_scenes_florence2`)**: Mô tả chi tiết nội dung từng slide keyframe bằng mô hình **Florence-2** (`florence2_vendor`).
3. **PaddleOCR (`extract_ocr_paddleocr`)**: Trích xuất toàn bộ chữ trên slide (tiếng Việt/tiếng Anh).

In [ ]:
from ai_workers.modules.visual_v2.semantic import SemanticAnalyzer

print("=== STAGE 4: SEMANTIC ANALYZER (CLIP + FLORENCE-2 + PADDLEOCR) ===")
semantic_analyzer = SemanticAnalyzer()
filtered_scenes = semantic_analyzer.process(visual_result.get("scenes", []))
visual_result["scenes"] = filtered_scenes
slides = filtered_scenes

print(f"Filtered Distinct Slides: {len(slides)}")
for slide in slides[:3]:
    print(f"  Slide at {slide.get('start_timecode')}:")
    print(f"    OCR Preview: {slide.get('ocr_text', '')[:60]}...")
    print(f"    Florence-2 Caption: {slide.get('caption', '')}")

### 6.1 [VẤN ĐỀ 7] Visual Quality Gate & OCR-Aware Smart Slide Deduplication
*Mục đích*: Loại bỏ ảnh mờ nhòe (Laplacian blur detection) và cải tiến thuật toán lọc trùng slide. Nếu 2 slide có hình thức tương tự (Cosine Similarity cao) nhưng OCR chứa nội dung chữ khác biệt (ví dụ slide cùng template nhưng cập nhật ý mới), **bắt buộc giữ lại cả 2 slide** thay vì xoá nhầm.

In [ ]:
import cv2
import numpy as np
from difflib import SequenceMatcher

def calculate_text_similarity(t1: str, t2: str) -> float:
    """Tính độ tương đồng văn bản giữa 2 đoạn OCR text."""
    if not t1 and not t2:
        return 1.0
    if not t1 or not t2:
        return 0.0
    return SequenceMatcher(None, t1.lower(), t2.lower()).ratio()

def smart_visual_quality_gate(
    scenes: list,
    min_blur_var: float = 40.0,
    cosine_thresh: float = 0.88,
    ocr_diff_thresh: float = 0.35,
    min_keep_ratio: float = 0.25,
) -> list:
    """
    Quality Gate cho Visual Pipeline (Giải quyết Vấn đề 7):
    1. Blur Detection (Laplacian Variance) + OCR rescue rule.
    2. OCR-Aware Deduplication: Không xoá slide nếu OCR chứa nội dung chữ khác biệt dù Cosine Sim cao.
    3. Ưu tiên giữ frame nét hơn khi duplicate và nội dung tương đương.
    4. Safety floor: luôn giữ tối thiểu một tỷ lệ keyframe để tránh collapse visual evidence.
    """
    if not scenes:
        return []

    processed_scenes = []
    blur_flagged = 0
    blur_removed = 0
    dedup_removed = 0
    ocr_saved_slides = 0

    for sc in scenes:
        path = sc.get("keyframe_path")
        blur_var = 100.0  # Default fallback
        if path and os.path.exists(path):
            img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                blur_var = cv2.Laplacian(img, cv2.CV_64F).var()

        ocr = sc.get("ocr_text", "").strip()
        sc["blur_score"] = round(float(blur_var), 2)
        sc["is_blurry"] = blur_var < min_blur_var
        sc["visual_gate_status"] = "kept"

        if sc["is_blurry"]:
            blur_flagged += 1
            # Nếu mờ và cũng không có OCR evidence thì loại bỏ luôn
            if len(ocr) < 8:
                sc["visual_gate_status"] = "dropped_blur_no_evidence"
                blur_removed += 1
                continue

        # Lọc trùng nâng cao dựa trên cả Embedding + OCR Text
        is_duplicate = False
        emb = sc.get("embedding")

        for kept in processed_scenes:
            kept_emb = kept.get("embedding")
            kept_ocr = kept.get("ocr_text", "").strip()

            visual_sim = 0.0
            if emb is not None and kept_emb is not None:
                visual_sim = float(np.dot(emb, kept_emb) / (np.linalg.norm(emb) * np.linalg.norm(kept_emb) + 1e-8))

            ocr_sim = calculate_text_similarity(ocr, kept_ocr)

            if visual_sim > cosine_thresh:
                # Nếu hình giống nhưng OCR khác đáng kể -> giữ lại (chống mất slide cập nhật nội dung)
                if ocr and kept_ocr and ocr_sim < (1.0 - ocr_diff_thresh):
                    ocr_saved_slides += 1
                    continue

                # Nếu cả 2 gần như cùng nội dung -> ưu tiên frame nét hơn
                if sc["blur_score"] > kept.get("blur_score", 0):
                    kept["visual_gate_status"] = "replaced_by_sharper_frame"
                    processed_scenes.remove(kept)
                    break

                sc["visual_gate_status"] = "dropped_duplicate"
                is_duplicate = True
                dedup_removed += 1
                break

        if not is_duplicate:
            processed_scenes.append(sc)

    min_keep = max(1, int(len(scenes) * min_keep_ratio))
    if len(processed_scenes) < min_keep:
        # Restore top sharp frames to keep minimum visual evidence coverage.
        ranked = sorted(
            scenes,
            key=lambda x: (x.get("blur_score", 0), len((x.get("ocr_text") or "").strip())),
            reverse=True,
        )
        processed_scenes = ranked[:min_keep]

    print("=== [VẤN ĐỀ 7] VISUAL QUALITY GATE & DEDUPLICATION REPORT ===")
    print(f"Keyframes ban đầu: {len(scenes)} -> Sau Quality Gate: {len(processed_scenes)}")
    print(f"Keyframes bị cảnh báo mờ (Blur): {blur_flagged}")
    print(f"Keyframes bị loại do mờ + thiếu OCR evidence: {blur_removed}")
    print(f"Keyframes bị loại do duplicate: {dedup_removed}")
    print(f"Slide được cứu nhờ khác biệt OCR (OCR-Saved Slides): {ocr_saved_slides}")
    print(f"Safety floor (min_keep_ratio={min_keep_ratio}): {min_keep}")
    return processed_scenes

if ENABLE_EXPERIMENTAL_GATES:
    quality_slides = smart_visual_quality_gate(
        slides,
        min_blur_var=30.0,
        cosine_thresh=0.88,
        ocr_diff_thresh=0.30,
        min_keep_ratio=MIN_KEYFRAME_KEEP_RATIO,
    )
else:
    quality_slides = slides
    print("[PARITY MODE] Visual Quality Gate bypassed to match backend pipeline.")

visual_result["scenes"] = quality_slides
slides = quality_slides

### 6.2 [VẤN ĐỀ 8] Caption Hallucination Verification & Evidence Grounding
*Mục đích*: Kiểm tra đối chiếu Caption được sinh bởi Vision Model (Florence-2) với văn bản OCR thực tế (PaddleOCR). Phát hiện các caption hallucinated (chứa nội dung không xuất hiện trên slide hoặc lặp từ vô nghĩa), đồng thời xây dựng `grounded_slide_context` để định hướng LLM chỉ sử dụng bằng chứng có thực.

In [ ]:
import re

def verify_and_ground_captions(scenes: list, min_grounding_score: float = 0.22) -> list:
    """
    Caption Verification & Hallucination Mitigation (Giải quyết Vấn đề 8):
    - Kiểm tra lặp từ (repetitive phrase loop) trong caption.
    - So sánh overlap từ khóa giữa Caption và OCR Text.
    - Gắn cờ grounded_status cho từng slide.
    - Xây dựng prompt context kèm Ground Truth OCR để chống LLM hallucination.
    """
    if not scenes:
        return []

    hallucination_count = 0
    low_grounding_count = 0

    for sc in scenes:
        caption = sc.get("caption", "").strip()
        ocr_text = sc.get("ocr_text", "").strip()

        is_hallucinated = False
        hallucination_reason = None

        # 1. Repetitive text check (ví dụ: "a photo of a photo of a photo...")
        words = caption.lower().split()
        if len(words) > 6:
            unique_ratio = len(set(words)) / len(words)
            if unique_ratio < 0.4:
                is_hallucinated = True
                hallucination_reason = "Repetitive word loop detected in VLM caption."

        # 2. OCR vs Caption Keyword Grounding Check
        ocr_words = set(re.findall(r'\w+', ocr_text.lower()))
        cap_words = set(re.findall(r'\w+', caption.lower()))

        # Loại bỏ stop words cơ bản
        stopwords = {"a", "an", "the", "is", "are", "of", "on", "in", "and", "with", "this", "image", "shows", "slide"}
        cap_keywords = cap_words - stopwords

        grounded_keywords = cap_keywords.intersection(ocr_words)
        grounding_score = len(grounded_keywords) / max(1, len(cap_keywords))

        # 3. Grounding threshold check: caption ít bám OCR evidence -> coi là rủi ro
        if cap_keywords and grounding_score < min_grounding_score:
            is_hallucinated = True
            if not hallucination_reason:
                hallucination_reason = "Low OCR grounding score."
            low_grounding_count += 1

        sc["grounding_score"] = round(grounding_score, 2)
        sc["is_hallucinated"] = is_hallucinated
        sc["hallucination_reason"] = hallucination_reason

        if is_hallucinated:
            hallucination_count += 1
            # Fallback caption bằng OCR preview nếu caption bị hỏng/hallucinate
            sc["verified_caption"] = f"Slide Text: {ocr_text[:120]}" if ocr_text else "Lecture Slide"
            sc["grounded_status"] = "fallback_to_ocr"
        else:
            sc["verified_caption"] = caption
            sc["grounded_status"] = "trusted"

        # Xây dựng Grounded Prompt Context cho LLM Summarizer ở Stage 6
        sc["llm_context_block"] = (
            f"[SLIDE KEYFRAME @ {sc.get('start_timecode')}]\n"
            f"- OCR Content (Primary Evidence): {ocr_text if ocr_text else '[No Text Detected]'}\n"
            f"- Visual Description: {sc['verified_caption']} (Grounding Score: {grounding_score:.2f})\n"
            f"- Grounded Status: {sc['grounded_status']}"
        )

    print("=== [VẤN ĐỀ 8] CAPTION HALLUCINATION VERIFICATION REPORT ===")
    print(f"Tổng số Slide được kiểm tra: {len(scenes)}")
    print(f"Số Caption nghi ngờ Hallucination / Lỗi: {hallucination_count}")
    print(f"Số slide có grounding score thấp: {low_grounding_count}")
    for sc in scenes[:3]:
        print(f"  Slide [{sc.get('start_timecode')}]: Grounding Score = {sc.get('grounding_score')} | Verified Cap: {sc.get('verified_caption')[:50]}...")

    return scenes

if ENABLE_EXPERIMENTAL_GATES:
    slides = verify_and_ground_captions(slides, min_grounding_score=0.22)
else:
    for sc in slides:
        sc["verified_caption"] = sc.get("caption", "")
        sc["grounded_status"] = "bypassed_parity"
        sc["grounding_score"] = 1.0
        sc["is_hallucinated"] = False
        sc["hallucination_reason"] = None
        sc["llm_context_block"] = ""
    print("[PARITY MODE] Caption grounding verification bypassed to match backend pipeline.")

## 7. Stage 5: Multimodal Timeline Alignment & Chaptering

In [ ]:
from ai_workers.modules.fusion.timeline import TimelineBuilder
import re

print("=== STAGE 5: TIMELINE BUILDER ===")

def clean_transcript_text(text: str, lang: str = "en") -> str:
    if not text:
        return ""

    # Language-aware filler removal to avoid over-cleaning domain terms.
    fillers_by_lang = {
        "vi": [r"\bà\b", r"\bừm\b", r"\bờ\b"],
        "en": [r"\buhm\b", r"\bumm\b", r"\bhmm\b", r"\byou know\b"],
    }
    fillers = fillers_by_lang.get(lang, fillers_by_lang["en"])

    cleaned = text
    for f in fillers:
        cleaned = re.sub(f, "", cleaned, flags=re.IGNORECASE)
    # Keep punctuation spacing stable for sentence models and chaptering.
    cleaned = re.sub(r"\s+([,.;:!?])", r"\1", cleaned)
    return re.sub(r"\s+", " ", cleaned).strip()

def post_process_utterances(utterances_list: list, min_dur: float = 1.5, lang: str = "en") -> list:
    if not utterances_list:
        return []
    processed = []
    for utt in utterances_list:
        c_text = clean_transcript_text(utt.get("text", ""), lang=lang)
        if not c_text:
            continue

        dur = utt.get("end", 0.0) - utt.get("start", 0.0)
        if processed and dur < min_dur and processed[-1].get("speaker") == utt.get("speaker"):
            processed[-1]["end"] = utt["end"]
            processed[-1]["text"] = (processed[-1].get("text", "") + " " + c_text).strip()
        else:
            item = dict(utt)
            item["text"] = c_text
            processed.append(item)
    return processed

# Hậu xử lý utterance trước Timeline để tránh lệch giữa dữ liệu vào LLM và dữ liệu export.
if ENABLE_UPSTREAM_UTTERANCE_CLEANUP:
    pipeline_utterances = post_process_utterances(
        utterances,
        min_dur=MIN_UTTERANCE_MERGE_DUR_SEC,
        lang=EXPECTED_LANG,
    )
else:
    pipeline_utterances = utterances

pipeline_slides = slides
print(f"Utterances raw -> pipeline: {len(utterances)} -> {len(pipeline_utterances)}")

timeline_builder = TimelineBuilder()
timeline_result = timeline_builder.process(
    pipeline_utterances,
    visual_result.get("scenes", []),
    pipeline_slides
)

chapters = timeline_result.get("chapters", [])
print(f"Total Chapters Formed: {len(chapters)}")
for idx, ch in enumerate(chapters, 1):
    start = ch.get("startTime", ch.get("start_time", ch.get("start_seconds")))
    end = ch.get("endTime", ch.get("end_time", ch.get("end_seconds")))
    print(f"  Chapter {idx}: {ch.get('title', 'Untitled')} ({start} - {end})")


## 8. Stage 6: Evidence-Grounded LLM Summarization (Truyền Bằng Chứng Thực Cho LLM)

In [ ]:
from ai_workers.modules.fusion.summarizer import Summarizer

print("=== STAGE 6: EVIDENCE-GROUNDED SUMMARIZER ===")
summarizer = Summarizer()

def prepare_inputs_for_summarizer(raw_utterances: list, raw_slides: list, audio_report: dict) -> tuple[list, list]:
    """
    Áp dụng policy xử lý thực tế trước khi vào LLM:
    - Vấn đề 5: nếu ASR yếu thì giảm phụ thuộc audio, lọc utterance nhiễu.
    - Vấn đề 7/8: luôn dùng verified_caption + llm_context_block làm visual evidence.
    """
    safe_slides = []
    for s in raw_slides:
        item = dict(s)
        if "llm_context_block" in item:
            item["caption"] = item.get("verified_caption", item.get("caption"))
        safe_slides.append(item)

    safe_utterances = []
    for utt in raw_utterances:
        txt = utt.get("text", "").strip()
        if len(txt) < 2:
            continue
        safe_utterances.append(utt)

    # Audio yếu -> lọc thêm những utterance quá ngắn/không đủ thông tin
    if audio_report.get("visual_dominant_fallback"):
        safe_utterances = [u for u in safe_utterances if len(u.get("text", "").strip()) >= 8]

    if not safe_utterances:
        safe_utterances = raw_utterances

    return safe_utterances, safe_slides

summ_utterances, summ_slides = prepare_inputs_for_summarizer(pipeline_utterances, pipeline_slides, audio_quality_report)
print(f"Summarizer mode: {audio_quality_report.get('recommended_mode')} | utterances={len(summ_utterances)} | slides={len(summ_slides)}")

text_result = summarizer.process(
    summ_utterances,
    summ_slides,
    chapters
)

print(f"Video Title: {text_result.get('video_title')}")
print(f"Model Used: {text_result.get('model_used')}")
print("\n--- Executive Summary Preview ---")
print(text_result.get("summary", "")[:400] + "...")

## 9. Stage 7: Backend Post-Pipeline Alignment (`ai_workers/tasks.py` Standard)

In [ ]:
print("=== STAGE 7: BACKEND POST-PIPELINE ALIGNMENT ===")
print("=== PIPELINE PARITY CHECK (Backend vs Experiment) ===")
print(f"- Audio Quality Gate enabled: {ENABLE_EXPERIMENTAL_GATES}")
print(f"- Visual Quality Gate enabled: {ENABLE_EXPERIMENTAL_GATES}")
print(f"- Caption Grounding Gate enabled: {ENABLE_EXPERIMENTAL_GATES}")
print(f"- Upstream Utterance Cleanup enabled: {ENABLE_UPSTREAM_UTTERANCE_CLEANUP}")
if ENABLE_EXPERIMENTAL_GATES or ENABLE_UPSTREAM_UTTERANCE_CLEANUP:
    print("[INFO] Current run = experimental-enhanced pipeline")
else:
    print("[INFO] Current run = backend-parity pipeline")

# 1. Align transcript utterances with visual scenes to populate scene "script"
current_scenes = visual_result.get("scenes", [])
scene_utterance_lists = {id(sc): [] for sc in current_scenes}

aligned_segments = timeline_result.get("aligned_segments", [])
if aligned_segments:
    for item in aligned_segments:
        scene_id = item.get("scene_id")
        utt_text = item.get("utterance", {}).get("text", "")
        if scene_id in scene_utterance_lists and utt_text:
            scene_utterance_lists[scene_id].append(utt_text)
else:
    # Fallback to pure temporal overlap
    for utt in pipeline_utterances:
        u_start = utt.get("start", 0.0)
        u_end = utt.get("end", 0.0)
        best_scene = None
        max_overlap = 0.0
        for sc in current_scenes:
            sc_start = sc.get("start_seconds", 0.0)
            sc_end = sc.get("end_seconds", 0.0)
            overlap = min(u_end, sc_end) - max(u_start, sc_start)
            if overlap > max_overlap:
                max_overlap = overlap
                best_scene = sc
        if best_scene is not None and max_overlap > 0:
            scene_utterance_lists[id(best_scene)].append(utt.get("text", ""))

for sc in current_scenes:
    sc["script"] = " ".join(scene_utterance_lists[id(sc)]).strip()
    if "keyframe_url" not in sc or not sc["keyframe_url"]:
        # Local fallback url matching backend static route
        fname = os.path.basename(sc.get("keyframe_path", "slide1.png"))
        sc["keyframe_url"] = f"/static/mock_r2/keyframes/{fname}"

# 2. Construct keyframes list for Frontend compatibility
keyframes_fe = []
for sc in current_scenes:
    keyframes_fe.append({
        "timestamp": sc.get("start_seconds", 0.0),
        "imageUrl": sc.get("keyframe_url"),
        "description": sc.get("verified_caption", sc.get("caption", f"Slide at {sc.get('start_timecode')}")),
        "transcript": sc.get("script", ""),
        "importanceScore": sc.get("importanceScore", 0.8)
    })

final_task_result = {
    "job_id": JOB_ID,
    "status": "done",
    "video_title": text_result.get("video_title", "Untitled Lecture Video"),
    "summary": text_result.get("summary", ""),
    "chapters": text_result.get("chapters", []),
    "keyframes": keyframes_fe,
    "transcript_text": " ".join([u.get("text", "") for u in pipeline_utterances]).strip(),
    "transcript_segments": pipeline_utterances,
    "scenes": current_scenes,
    "duration": pipeline_utterances[-1]["end"] if pipeline_utterances else 0.0,
    "model_used": text_result.get("model_used", "Groq"),
    "audio_quality_report": audio_quality_report,
    "processing_time": 12.5
}

print(f"Backend Task Result constructed! Keys: {list(final_task_result.keys())}")
print(f"Total Keyframes for FE: {len(final_task_result['keyframes'])}")

## 10. Stage 8: Backend Database & Vector Ingestion Simulation

In [ ]:
print("=== STAGE 8: BACKEND DB & CHROMADB INGESTION SIMULATION ===")

# 1. Simulation of Summary SQL DB Record
summary_db_record = {
    "video_id": "mock_video_123",
    "summary_text": final_task_result.get("summary"),
    "chapters_json": final_task_result.get("chapters"),
    "keyframes_json": final_task_result.get("keyframes"),
    "transcript_text": final_task_result.get("transcript_text"),
    "model_used": final_task_result.get("model_used"),
    "processing_time": final_task_result.get("processing_time")
}

# 2. Simulation of VideoScene SQL DB Records
video_scene_records = []
for sc in final_task_result.get("scenes", []):
    video_scene_records.append({
        "video_id": "mock_video_123",
        "scene_index": sc.get("scene_index"),
        "start_seconds": sc.get("start_seconds"),
        "end_seconds": sc.get("end_seconds"),
        "start_timecode": sc.get("start_timecode"),
        "end_timecode": sc.get("end_timecode"),
        "keyframe_path": sc.get("keyframe_path"),
        "keyframe_url": sc.get("keyframe_url"),
        "caption": sc.get("verified_caption", sc.get("caption")),
        "script": sc.get("script")
    })

# 3. Simulation of ChromaDB Vector Ingestion
vector_chunks = []
vector_metadatas = []
for idx, seg in enumerate(final_task_result.get("transcript_segments", [])):
    vector_chunks.append(seg.get("text", ""))
    vector_metadatas.append({
        "video_id": "mock_video_123",
        "chunk_index": idx,
        "timestamp_start": float(seg.get("start", 0.0))
    })

print(f"[OK] Simulated SQL Summary Record created.")
print(f"[OK] Simulated {len(video_scene_records)} VideoScene Records created.")
print(f"[OK] Prepared {len(vector_chunks)} vector chunks for ChromaDB ingestion.")

## 11. Stage 9: Integrated Post-Processing & Quality Assurance Suite

### 9.1 Hậu xử lý Lời thoại (Clean Fillers & Short Utterances Merging)
*Mục đích*: Làm sạch từ đệm và gộp phân đoạn ngắn TRƯỚC khi gửi cho LLM & ChromaDB để tăng chất lượng tóm tắt.

In [ ]:
# Đã hậu xử lý utterances từ Stage 5 để đồng bộ dữ liệu cho Timeline + Summarizer + Export.
post_utterances = pipeline_utterances
print(f"Utterances ban đầu: {len(utterances)} -> Sau hậu xử lý (đã áp dụng upstream): {len(post_utterances)}")

### 9.2 [VẤN ĐỀ 7] Smart Slide Deduplication (OCR-Aware + Cosine Threshold)

In [ ]:
# Tránh chạy dedup lần 2 sau khi đã summarize để không gây data drift giữa nội dung LLM và export.
post_slides = current_scenes
print(f"Slides ban đầu: {len(current_scenes)} -> Sau hậu xử lý: {len(post_slides)} (giữ nguyên để đồng bộ pipeline)")

### 9.3 Hậu xử lý Chương Mục (Chapter Boundary Smoothing & Minimum Duration Filtering)

In [ ]:
def _chapter_start(ch: dict) -> float:
    for k in ("startTime", "start_time", "start_seconds", "start"):
        if k in ch and ch[k] is not None:
            return float(ch[k])
    return 0.0


def _chapter_end(ch: dict) -> float:
    for k in ("endTime", "end_time", "end_seconds", "end"):
        if k in ch and ch[k] is not None:
            return float(ch[k])
    return _chapter_start(ch) + 10.0


def post_process_chapters(chapter_list: list, min_dur_sec: float = 45.0) -> list:
    """Merge short chapters. Supports timeline/LLM schemas: startTime/endTime or start_seconds/end_seconds."""
    if not chapter_list:
        return []
    smoothed = []
    for idx, ch in enumerate(chapter_list, 1):
        item = dict(ch)
        s_sec = _chapter_start(item)
        e_sec = _chapter_end(item)
        if e_sec <= s_sec:
            e_sec = s_sec + 10.0

        item["title"] = item.get("title") or f"Chapter {idx}"
        item["summary"] = item.get("summary") or ""
        item["startTime"] = s_sec
        item["endTime"] = e_sec
        item["start_seconds"] = s_sec
        item["end_seconds"] = e_sec
        item["start_time"] = s_sec
        item["end_time"] = e_sec

        dur = e_sec - s_sec
        if smoothed and dur < min_dur_sec:
            smoothed[-1]["endTime"] = e_sec
            smoothed[-1]["end_time"] = e_sec
            smoothed[-1]["end_seconds"] = e_sec
            if item.get("summary"):
                smoothed[-1]["summary"] = (smoothed[-1].get("summary", "") + " " + item["summary"]).strip()
            if item.get("title") and not smoothed[-1].get("title"):
                smoothed[-1]["title"] = item["title"]
        else:
            smoothed.append(item)
    return smoothed

# Prefer LLM-enriched chapters (title/summary); fallback to timeline boundaries
chapters_for_post = text_result.get("chapters") or final_task_result.get("chapters") or chapters
post_chapters = post_process_chapters(chapters_for_post, min_dur_sec=MIN_CHAPTER_DURATION_SEC)
print(f"Chương ban đầu: {len(chapters_for_post)} -> Sau hậu xử lý làm mịn: {len(post_chapters)}")
for idx, ch in enumerate(post_chapters, 1):
    print(f"  [{idx}] {ch.get('start_time'):.1f}s-{ch.get('end_time'):.1f}s | {ch.get('title', 'Untitled')}")


### 9.4 Multimodal Image & Slide Retrieval (CLIP Embeddings)

In [ ]:
if ENABLE_RETRIEVAL_TESTS:
    import numpy as np
    import torch
    from PIL import Image
    from transformers import CLIPProcessor, CLIPModel

    def retrieve_slides_by_text(query_text: str, scene_list: list, top_k: int = 3) -> list:
        """Truy vấn Slide keyframe bài giảng theo câu tìm kiếm bằng CLIP Text Encoder."""
        if not scene_list:
            return []

        device = "cuda" if torch.cuda.is_available() else "cpu"
        clip_local = os.path.join(str(PROJECT_ROOT), "cache", "clip-vit-base-patch32")
        clip_source = clip_local if os.path.isdir(clip_local) else "openai/clip-vit-base-patch32"
        processor = CLIPProcessor.from_pretrained(
            clip_source,
            local_files_only=os.path.isdir(clip_local),
        )
        model = CLIPModel.from_pretrained(
            clip_source,
            local_files_only=os.path.isdir(clip_local),
        ).to(device)

        inputs = processor(text=[query_text], return_tensors="pt", padding=True).to(device)
        with torch.no_grad():
            query_feat = model.get_text_features(**inputs)
            query_emb = (query_feat / query_feat.norm(dim=-1, keepdim=True)).cpu().numpy()[0]

        scores = []
        for sc in scene_list:
            emb = sc.get("embedding")
            if emb is not None:
                sim = float(
                    np.dot(query_emb, emb)
                    / (np.linalg.norm(query_emb) * np.linalg.norm(emb) + 1e-8)
                )
                scores.append((sim, sc))

        scores.sort(key=lambda x: x[0], reverse=True)
        return scores[:top_k]

    # Thử nghiệm Image / Slide Retrieval theo từ khóa
    query = "neural network diagram architecture"
    search_results = retrieve_slides_by_text(query, current_scenes, top_k=3)

    print(f"=== TRUY VẤN SLIDE KEYFRAME VỚI QUERY: '{query}' ===")
    for rank, (score, sc) in enumerate(search_results, 1):
        print(f"Rank #{rank} [Score: {score:.4f}] - Slide at {sc.get('start_timecode')}")
        print(f"  Verified Caption: {sc.get('verified_caption', sc.get('caption'))}")
        print(f"  Script Preview: {sc.get('script', '')[:80]}...")
else:
    print("[SKIP] Stage 9.4 retrieval test disabled in compare run.")

### 9.5 Production-Ready Export kèm Báo Cáo QA 5-7-8

In [ ]:
def export_backend_production_summary(task_result: dict, chapters: list, slides: list, save_path: str):
    content = []
    content.append(f"# {task_result.get('video_title', 'Lecture Summary')}\n")
    content.append(f"- **Job ID:** {task_result.get('job_id')}")
    content.append(f"- **Model:** {task_result.get('model_used')}")
    content.append(f"- **Duration:** {task_result.get('duration', 0):.2f} seconds\n")
    
    # Quality Assurance Summary Report (Vấn đề 5, 7, 8)
    aq = task_result.get("audio_quality_report", {})
    content.append("## 0. Quality Assurance & Quality Gate Report (Problems 5, 7, 8)\n")
    content.append(f"- **[Vấn đề 5] Audio Quality Status:** {aq.get('quality_status', 'N/A')} (Avg Conf: {aq.get('average_confidence', 0):.2f})")
    content.append(f"- **[Vấn đề 7] Visual Keyframes Kept:** {len(slides)} (Blur & OCR-Aware Deduplicated)")
    content.append(f"- **[Vấn đề 8] Caption Grounding Status:** Checked against PaddleOCR Evidence\n")
    
    content.append("## 1. Executive Summary\n")
    content.append(f"{task_result.get('summary')}\n")
    content.append("## 2. Table of Contents / Chapters\n")
    for idx, ch in enumerate(chapters, 1):
        start = ch.get("startTime", ch.get("start_time", ch.get("start_seconds", "?")))
        end = ch.get("endTime", ch.get("end_time", ch.get("end_seconds", "?")))
        title = ch.get("title") or f"Chapter {idx}"
        content.append(f"### [{start} - {end}] {title}")
        if ch.get("summary"):
            content.append(f"{ch.get('summary')}\n")
    content.append("## 3. Visual Keyframes & Slide Scripts\n")
    for sc in slides:
        cap = sc.get("verified_caption", sc.get("caption", "Keyframe"))
        content.append(f"- **Slide [{sc.get('start_timecode')}]**: {cap}")
        if sc.get("ocr_text"):
            content.append(f"  - *OCR Text*: {sc.get('ocr_text')[:100]}...")
        if sc.get("script"):
            content.append(f"  - *Utterance Script*: {sc.get('script')[:120]}...")
            
    final_md = "\n".join(content)
    with open(save_path, "w", encoding="utf-8") as f:
        f.write(final_md)
    print(f"[OK] Exported production summary with QA Report to: {save_path}")
    return final_md

export_file = os.path.join(OUTPUT_DIR, "production_lecture_summary.md")
export_backend_production_summary(final_task_result, post_chapters, post_slides, export_file)
